In [87]:
# Importing the dependecies

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from wordcloud import WordCloud
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to C:\Users\Subham
[nltk_data]     Pathak\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Subham
[nltk_data]     Pathak\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [88]:
df = pd.read_csv(r'C:\Users\Subham Pathak\Desktop\AI\MLops\ML_Pipeline_AWS-DVC\spam.csv')

df.head(10)

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
5,spam,FreeMsg Hey there darling it's been 3 week's n...
6,ham,Even my brother is not like to speak with me. ...
7,ham,As per your request 'Melle Melle (Oru Minnamin...
8,spam,WINNER!! As a valued network customer you have...
9,spam,Had your mobile 11 months or more? U R entitle...


In [89]:
df = df[['Message', 'Category']]


In [90]:
df = df.rename(columns={'Message': 'Text', 'Category': 'Target'})

In [91]:
df

,Text,Target
0,"Go until jurong point, crazy.. Available only ...",ham
1,Ok lar... Joking wif u oni...,ham
2,Free entry in 2 a wkly comp to win FA Cup fina...,spam
3,U dun say so early hor... U c already then say...,ham
4,"Nah I don't think he goes to usf, he lives aro...",ham
...,...,...
5567,This is the 2nd time we have tried 2 contact u...,spam
5568,Will ü b going to esplanade fr home?,ham
5569,"Pity, * was in mood for that. So...any other s...",ham
5570,The guy did some bitching but I acted like i'd...,ham


### **DATA PREPROCESSING**

In [92]:
df.duplicated().sum()

np.int64(415)

In [93]:
df.isnull().sum()

Text      0
Target    0
dtype: int64

In [94]:
df.drop_duplicates(keep='first')

,Text,Target
0,"Go until jurong point, crazy.. Available only ...",ham
1,Ok lar... Joking wif u oni...,ham
2,Free entry in 2 a wkly comp to win FA Cup fina...,spam
3,U dun say so early hor... U c already then say...,ham
4,"Nah I don't think he goes to usf, he lives aro...",ham
...,...,...
5567,This is the 2nd time we have tried 2 contact u...,spam
5568,Will ü b going to esplanade fr home?,ham
5569,"Pity, * was in mood for that. So...any other s...",ham
5570,The guy did some bitching but I acted like i'd...,ham


### **Feature Engineering**

In [95]:
from nltk.stem.porter import PorterStemmer
import string
ps = PorterStemmer()

In [96]:
def text_transform(text):
    
    text=text.lower()
    text = nltk.word_tokenize(text)
        
    y = []
    
    for i in text:
        if i.isalnum():
            y.append(i)
            
    text = y[:]
    y.clear()
    
    for i in text:
        if i not in stopwords.words('english') and i not in string.punctuation:
            y.append(i)
    
    text = y[:]
    y.clear()
    
    for i in text:
        y.append(ps.stem(i))
        
    return " ".join(y)        

In [97]:
df['transformed_text'] = df['Text'].apply(text_transform)

In [98]:
df.head()

,Text,Target,transformed_text
0,"Go until jurong point, crazy.. Available only ...",ham,go jurong point crazi avail bugi n great world...
1,Ok lar... Joking wif u oni...,ham,ok lar joke wif u oni
2,Free entry in 2 a wkly comp to win FA Cup fina...,spam,free entri 2 wkli comp win fa cup final tkt 21...
3,U dun say so early hor... U c already then say...,ham,u dun say earli hor u c alreadi say
4,"Nah I don't think he goes to usf, he lives aro...",ham,nah think goe usf live around though


In [99]:
encoder = LabelEncoder()

df['target'] = encoder.fit_transform(df['Target'])

In [100]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
tfid = TfidfVectorizer(max_features=600)

In [101]:
x = tfid.fit_transform(df['transformed_text'])
y = df['target'].values

### **Train Test Split**

In [102]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, random_state=41, stratify = y, test_size=0.2)
 

#### **Model Training**

In [103]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, BaggingClassifier, GradientBoostingClassifier, ExtraTreesClassifier

In [104]:
svc = SVC(kernel='sigmoid', gamma=1.0)
knc = KNeighborsClassifier()
mnt = MultinomialNB()
dct = DecisionTreeClassifier(max_depth=5)
lrc = LogisticRegression(solver='liblinear', penalty='l1')
rfc = RandomForestClassifier(n_estimators=50, random_state=2)
abc = AdaBoostClassifier(n_estimators=50, random_state=2)
bgc = BaggingClassifier(n_estimators=50, random_state=2)
gbc = GradientBoostingClassifier(n_estimators=50, random_state=2)
etc = ExtraTreesClassifier(n_estimators=50, random_state=2)


In [124]:
clfs = {
    'SVC': svc,
    'KNN' : knc,
    'NB' : mnt,
    'DT': dct,
    'LR' : lrc,
    'RF': rfc,
    'Adaboost' : abc,
    'BGC' : bgc,
    'GBC' : gbc,
    'ETC' : etc
    
}

#### **Model Evaluation**

In [126]:
from sklearn.metrics import accuracy_score, precision_score

In [127]:
def train_classifier(clfs, x_train, y_train, x_test, y_test):
    
    clfs.fit(x_train, y_train)
    y_pred = clfs.predict(x_test)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    return accuracy, precision

In [128]:
accuracy_score = []
precision_score = []

for name, clfs in clfs.items():
    current_accuracy, current_precision = train_classifier(clfs, x_train, y_train, x_test, y_test)
    print()
    print(f'For {name}')
    print(f"Accuracy : {current_accuracy}")
    print(f"Precision : {current_precision}")
    
    accuracy_score.append(current_accuracy)
    precision_score.append(current_precision)

TypeError: 'list' object is not callable